# Sunspot Modeling — Implementation / 흑점 모델링 구현

**Paper**: Rempel, M. & Schlichenmaier, R., "Sunspot Modeling: From Simplified Models to Radiative MHD Simulations", *Living Reviews in Solar Physics*, **8**, 3 (2011). [DOI: 10.12942/lrsp-2011-3]

## 목적 / Purpose

이 노트북은 Rempel & Schlichenmaier (2011)의 핵심 물리 개념 전반을 Python으로 구현한다:

1. **Solar stratification** — 대류층 barotropic 모형, 압력/밀도 scale height
2. **Plasma β 깊이 의존성** — monolithic flux tube가 깊이에 따라 어떻게 $\beta$-dominant regime으로 전환되는지
3. **Stefan-Boltzmann 흑점 광도** — $T_u, T_p$ 에서 $L/L_{\text{QS}}$ 계산
4. **Alfvén 속도 vs. 깊이** — timestep 제약이 MHD 시뮬레이션에서 왜 심각한지
5. **Jahn–Schmidt tripartite 모델 (2D 축대칭)** — magnetostatic flux tube 형태 재현 (Fig. 1)
6. **Moving flux tube 간이 ODE** — Schlichenmaier (1998) 동적 모델의 축약 버전
7. **Magneto-convection regime 분류** — Chandrasekhar number $\zeta$ 와 oscillatory vs overturning
8. **Synthetic sunspot intensity** — umbra + penumbra + radial filament 구조 + 회색 granulation
9. **Evershed flow 프로파일** — 반경 함수 $v_{\text{Ev}}(r)$ 모델
10. **NCP 계산** — Gradient-bearing flow channel에서 Stokes V 적분
11. **Sunspot lifetime vs disconnection depth** — Rempel (2011c) scaling 재현

This notebook implements the key physical concepts of Rempel & Schlichenmaier (2011) in Python: convection-zone stratification, depth-dependent plasma $\beta$, Stefan–Boltzmann sunspot luminosity, Alfvén-speed depth profile, the Jahn–Schmidt tripartite magnetostatic model (Fig. 1), a simplified moving-flux-tube ODE, magneto-convection regimes via the Chandrasekhar number, a synthetic sunspot intensity map, an Evershed-flow radial profile, a net-circular-polarization (NCP) calculation, and a disconnection-depth vs. sunspot-lifetime scaling.

**Environment**: conda `study-with-ai`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

rng = np.random.default_rng(seed=42)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (8, 5)

# Physical constants (CGS).
G_cgs = 6.67430e-8        # gravitational constant [cm^3/g/s^2]
k_B = 1.380649e-16        # Boltzmann [erg/K]
m_p = 1.67262192e-24      # proton mass [g]
sigma_SB = 5.670374419e-5 # Stefan-Boltzmann [erg/cm^2/s/K^4]
R_sun = 6.957e10          # solar radius [cm]
M_sun = 1.989e33          # solar mass [g]
g_sun = G_cgs * M_sun / R_sun ** 2  # surface gravity [cm/s^2]
print(f'Surface gravity g = {g_sun:.2e} cm/s^2')

---
## 1. Solar stratification / 태양 대류층 층화

흑점 모델링에서 depth $z$에 따른 $p(z)$, $\rho(z)$, $T(z)$ 프로파일이 필수. 여기서는 간단한 **polytropic convection-zone model**을 사용한다:
$$T(z) = T_0 - \gamma_T z,\qquad p(z) = p_0\left(\frac{T(z)}{T_0}\right)^{(m+1)},\qquad\rho(z) = \rho_0\left(\frac{T(z)}{T_0}\right)^m$$
폴리트로픽 지수 $m=1.5$ (adiabatic) 사용. 실제 태양 대류층과는 정확히 일치하지 않지만 physics의 scaling을 잘 포착.

In [ ]:
def convection_zone_profile(z_cm, T0=6060.0, p0=1.2e5, rho0=2.5e-7,
                            gamma_T=1e-4, m_poly=1.5):
    """Return polytropic solar-convection-zone stratification.

    Uses a simple polytropic model T(z) = T0 - gamma_T z, and pressure/
    density follow from hydrostatic equilibrium with polytrope index m.

    Args:
        z_cm: Depth array below photosphere in cm (positive downward).
        T0: Photospheric temperature [K] (default 6060).
        p0: Photospheric gas pressure [dyn/cm^2].
        rho0: Photospheric mass density [g/cm^3].
        gamma_T: Super-adiabatic gradient in K/cm.
        m_poly: Polytrope index (1.5 = adiabatic).

    Returns:
        Dict with keys T [K], p [cgs], rho [cgs], H_p [cm].
    """
    T = T0 + gamma_T * z_cm  # increases with depth
    ratio = T / T0
    p = p0 * ratio ** (m_poly + 1)
    rho = rho0 * ratio ** m_poly
    H_p = k_B * T / (m_p * g_sun)  # isothermal pressure scale height [cm]
    return dict(T=T, p=p, rho=rho, H_p=H_p)


# Depth grid from photosphere to 15 Mm below (Jahn-Schmidt depth).
z_Mm = np.linspace(0, 15, 200)
z_cm = z_Mm * 1e8
prof = convection_zone_profile(z_cm)

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].plot(z_Mm, prof['T'])
ax[0].set_xlabel('Depth [Mm]'); ax[0].set_ylabel('T [K]')
ax[0].set_title('Temperature')
ax[1].semilogy(z_Mm, prof['p'])
ax[1].set_xlabel('Depth [Mm]'); ax[1].set_ylabel('p [dyn/cm^2]')
ax[1].set_title('Gas pressure')
ax[2].semilogy(z_Mm, prof['rho'])
ax[2].set_xlabel('Depth [Mm]'); ax[2].set_ylabel(r'$\rho$ [g/cm^3]')
ax[2].set_title('Mass density')
plt.tight_layout(); plt.show()

print(f'Photosphere H_p = {prof["H_p"][0]/1e5:.0f} km')
print(f'@ 15 Mm depth: H_p = {prof["H_p"][-1]/1e5:.0f} km')
print(f'@ 15 Mm depth: p = {prof["p"][-1]:.2e}, rho = {prof["rho"][-1]:.2e}')

---
## 2. Plasma $\beta$ 깊이 의존성 / Depth-dependent plasma beta

흑점 자속 보존: 깊이에 따라 area $A\propto B^{-1}$. Hydrostatic 외부 압력이 급증하므로, 자속관 내부의 $p$도 증가해야 하고 이에 따라 $B$도 증가한다. 하지만 $p$가 $B^2$보다 더 빠르게 증가하므로 **깊이에 따라 $\beta$가 증가**한다. 이것이 monolithic → spaghetti 전환의 물리적 근거.

In [ ]:
def flux_tube_beta(z_cm, B_surface=3000.0, conserve='flux'):
    """Compute plasma beta inside a monolithic flux tube.

    The simplest model assumes total (gas + magnetic) pressure inside the
    tube equals the ambient gas pressure, and magnetic flux is conserved.
    For a back-of-envelope calculation we take B(z) such that the internal
    gas pressure is a fixed fraction f of the ambient, and balance the rest
    via magnetic pressure.

    Args:
        z_cm: Depth array in cm.
        B_surface: Surface magnetic field strength in Gauss.
        conserve: 'flux' — B scales to maintain 50% gas-pressure deficit;
                   'const' — B is fixed at surface value.

    Returns:
        Dict with B [G], beta, p_in, p_out, area_ratio.
    """
    prof = convection_zone_profile(z_cm)
    p_out = prof['p']
    # Require internal gas pressure + magnetic pressure = external.
    # At surface, magnetic pressure = B_surface^2/(8 pi) ~ external pressure.
    p_mag_surface = B_surface ** 2 / (8 * np.pi)
    frac_gas = 1.0 - p_mag_surface / p_out[0]
    frac_gas = max(frac_gas, 0.0)
    p_in = frac_gas * p_out
    p_mag = p_out - p_in
    B = np.sqrt(8 * np.pi * p_mag)
    if conserve == 'const':
        B = np.full_like(z_cm, B_surface)
        p_mag = B ** 2 / (8 * np.pi)
    beta = p_in / p_mag
    # Area ratio from flux conservation A(z)*B(z) = const.
    area = B[0] / B  # normalised to surface
    return dict(B=B, beta=beta, p_in=p_in, p_out=p_out, area=area)


tube = flux_tube_beta(z_cm, B_surface=3000.0)

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].plot(z_Mm, tube['B'])
ax[0].set_xlabel('Depth [Mm]'); ax[0].set_ylabel('B [G]')
ax[0].set_title('Field strength inside tube')
ax[1].semilogy(z_Mm, tube['beta'])
ax[1].axhline(1, color='k', ls='--', lw=0.8, label=r'$\beta=1$')
ax[1].set_xlabel('Depth [Mm]'); ax[1].set_ylabel(r'$\beta = 8\pi p/B^2$')
ax[1].set_title('Plasma beta (depth)')
ax[1].legend()
ax[2].plot(z_Mm, np.sqrt(tube['area']))
ax[2].set_xlabel('Depth [Mm]'); ax[2].set_ylabel('Tube radius (relative)')
ax[2].set_title('Funneling: radius vs depth')
plt.tight_layout(); plt.show()

idx2 = np.argmin(np.abs(z_Mm - 2))
print(f'@ surface : B = {tube["B"][0]:.0f} G, beta = {tube["beta"][0]:.2f}, radius = {np.sqrt(tube["area"][0]):.2f}')
print(f'@ z=2 Mm  : B = {tube["B"][idx2]:.0f} G, beta = {tube["beta"][idx2]:.1f}')
print(f'@ z=15 Mm : B = {tube["B"][-1]:.0f} G, beta = {tube["beta"][-1]:.1f}, radius = {np.sqrt(tube["area"][-1]):.2f}')
print('\nJahn & Schmidt (1994): umbra radius doubles between 15 Mm and surface.')
print(f'Simulated radius ratio: {np.sqrt(tube["area"][-1])/np.sqrt(tube["area"][0]):.2f}')

---
## 3. Stefan-Boltzmann 흑점 광도 / Sunspot luminosity

$$\frac{L(T)}{L_{\text{QS}}} = \left(\frac{T}{T_{\text{QS}}}\right)^4$$

논문의 숫자 — $T_u=4000$ K, $T_p=5275$ K, $T_{\text{QS}}=6060$ K → umbra ~19%, penumbra ~58%.

In [ ]:
def luminosity_fraction(T, T_QS=6060.0):
    """Stefan-Boltzmann luminosity fraction relative to quiet Sun."""
    return (T / T_QS) ** 4


T_arr = np.linspace(3500, 6100, 100)
L_arr = luminosity_fraction(T_arr)

plt.figure(figsize=(7, 4.5))
plt.plot(T_arr, L_arr * 100)
for name, T in [('Umbra (4000 K)', 4000), ('Penumbra (5275 K)', 5275),
                ('Quiet Sun (6060 K)', 6060)]:
    L = luminosity_fraction(T) * 100
    plt.scatter(T, L, s=60, zorder=5)
    plt.annotate(f'{name}\n{L:.0f}%', xy=(T, L), xytext=(8, 4),
                 textcoords='offset points', fontsize=9)
plt.xlabel('Temperature [K]')
plt.ylabel(r'$L/L_{\rm QS}$ [%]')
plt.title('Stefan-Boltzmann sunspot luminosity')
plt.grid(alpha=0.3)
plt.show()

print(f'Umbra heat-flux deficit: {100*(1 - luminosity_fraction(4000)):.1f}%  (paper: 77%)')
print(f'Penumbra heat-flux deficit: {100*(1 - luminosity_fraction(5275)):.1f}%  (paper: ~25%)')

---
## 4. Alfvén 속도와 timestep 제약 / Alfvén speed and CFL

$$v_A = \frac{B}{\sqrt{4\pi\rho}}$$

깊이에 따라 $\rho$가 증가하지만 $B$는 상대적으로 천천히 감소하므로, **umbra 상부(낮은 밀도, 높은 $B$)에서 $v_A$가 폭발**한다. 이것이 MHD 시뮬레이션의 timestep 제약의 주범이다.

In [ ]:
def alfven_speed(B_G, rho_cgs):
    """Alfvén speed in km/s given B in Gauss and rho in g/cm^3."""
    return B_G / np.sqrt(4 * np.pi * rho_cgs) / 1e5


# Extend grid above photosphere (z<0).
z_full = np.linspace(-1, 15, 400) * 1e8  # cm; -1 Mm above to 15 Mm below
rho_full = np.piecewise(
    z_full,
    [z_full < 0, z_full >= 0],
    [lambda z: 2.5e-7 * np.exp(z / prof['H_p'][0]),  # exp drop above photosphere
     lambda z: convection_zone_profile(z)['rho']],
)
B_umbra = 3000.0  # constant in the upper layers for simplicity
v_A = alfven_speed(B_umbra, rho_full)

# CFL condition: dt <= dx / v_A.
dx_km = 16.0  # Rempel 2011b resolution
dt_ms = (dx_km * 1e5) / (v_A * 1e5) * 1e3  # ms

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].semilogy(z_full/1e8, v_A)
ax[0].axvline(0, color='k', ls=':', lw=0.8)
ax[0].set_xlabel('z [Mm] (negative = above photosphere)')
ax[0].set_ylabel(r'$v_A$ [km/s]')
ax[0].set_title('Alfvén speed for B=3 kG along umbra axis')
ax[0].grid(alpha=0.3)
ax[1].semilogy(z_full/1e8, dt_ms)
ax[1].axvline(0, color='k', ls=':', lw=0.8)
ax[1].set_xlabel('z [Mm]')
ax[1].set_ylabel(f'CFL dt [ms] for dx={dx_km} km')
ax[1].set_title('CFL timestep constraint')
ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Max Alfvén speed (near top): {v_A.max():.0f} km/s')
print(f'Min CFL timestep: {dt_ms.min()*1000:.1f} micros')
print('Paper note: v_A > few 1000 km/s above umbra → severe dt constraint → Lorentz-force limiter used.')

---
## 5. Jahn–Schmidt tripartite 모델 재현 / Reproduction of the tripartite model

Jahn & Schmidt (1994) — 2D 축대칭 magnetostatic model. 자기장 선의 형태를 두 current sheet(peripatopause at umbra/penumbra, magnetopause at penumbra/quiet Sun)로 분리된 **self-similar fan-out** 으로 근사한다. 여기서는 **analytical self-similar streamfunction**으로 Fig. 1의 field-line pattern을 재현한다.

In [ ]:
def tripartite_field_lines(r, z, R_u_surface=6.0, R_p_surface=18.0, z_bp_Mm=15.0):
    """Return self-similar streamfunction of tripartite sunspot model.

    Each field line is labelled by its radial position r_0 at the surface.
    Line radius at depth z scales as sqrt(A(z)/A(0)) where A expands with
    depth. We use a simple functional fit capturing the 'funnel' widening.

    Args:
        r: Radial coordinate grid in Mm.
        z: Depth grid in Mm (positive downward).
        R_u_surface: Umbra radius at the surface [Mm].
        R_p_surface: Penumbra outer radius at surface [Mm].
        z_bp_Mm: Depth where the magnetopause touches the axis [Mm].

    Returns:
        Streamfunction value on the (r,z) grid (constant along field lines).
    """
    # Self-similar funnel: line at surface r0 is at r = r0 * f(z).
    # f(z) grows by ~4x from surface to z_bp (area factor 4 → radius 2).
    f = 1.0 - 0.5 * (z / z_bp_Mm) * (z < z_bp_Mm) - 0.5 * (z >= z_bp_Mm)
    # r0 of the field line passing through (r,z):
    r0 = r / f
    return r0  # stream label = surface footpoint radius


r = np.linspace(0, 25, 400)
z = np.linspace(0, 18, 400)
R, Z = np.meshgrid(r, z)
psi = tripartite_field_lines(R, Z)

fig, ax = plt.subplots(figsize=(9, 6))
# Draw field lines as contours of the stream label.
levels = np.linspace(0.5, 24, 30)
ax.contour(R, -Z, psi, levels=levels, colors='tab:blue', linewidths=0.7)
# Highlight peripatopause (umbra-penumbra) and magnetopause.
ax.contour(R, -Z, psi, levels=[6.0, 18.0], colors=['tab:red', 'tab:orange'],
           linewidths=2.0)
ax.axhline(0, color='k', lw=1)
ax.text(3, 0.5, 'umbra', fontsize=11, ha='center')
ax.text(12, 0.5, 'penumbra', fontsize=11, ha='center')
ax.text(22, 0.5, 'quiet Sun', fontsize=11, ha='center')
ax.set_xlabel('radius r [Mm]')
ax.set_ylabel('height z [Mm]')
ax.set_title('Jahn–Schmidt tripartite model — self-similar field-lines\n'
             'red=peripatopause, orange=magnetopause')
ax.set_xlim(0, 25); ax.set_ylim(-17, 1)
plt.tight_layout(); plt.show()

---
## 6. Moving flux tube ODE (Schlichenmaier 1998 simplified)

얇은 자속관의 동역학: tube footpoint가 radiatively heated 되어 상승, 광구에서 냉각되어 horizontal outflow로 전환. Schlichenmaier (1998)의 full 방정식을 **근사화**하여 tube apex의 높이 $h(t)$ 와 속도 $v(t)$ 를 추적한다.
$$\dot h = v,\qquad \dot v = g\,\frac{\Delta\rho}{\rho} - \frac{v^2}{L} + \frac{1}{\rho}\frac{\partial p}{\partial s}$$
여기서 첫째 항은 buoyancy, 둘째는 drag, 셋째는 radiative heating-driven pressure gradient.

In [ ]:
def moving_tube_rhs(t, y, g=g_sun, L_cool=5e7, drho_over_rho=0.3,
                    heating_timescale=30.0):
    """ODE right-hand side for a simplified moving flux tube.

    State vector y = [h (cm), v (cm/s)]. Buoyancy drives rise; radiative
    cooling at the photosphere converts vertical motion into horizontal
    outflow by reducing drho/rho once h > 0.

    Args:
        t: Time in seconds.
        y: State vector [h_cm, v_cm_per_s].
        g: Gravity (cm/s^2).
        L_cool: Cooling length scale in cm.
        drho_over_rho: Initial density deficit (buoyancy driver).
        heating_timescale: Characteristic heating time (s).

    Returns:
        dy/dt [dh/dt, dv/dt].
    """
    h, v = y
    # Density deficit decays once tube is cooled at photosphere.
    cooled = np.clip(h / L_cool, 0.0, 1.0)
    drho = drho_over_rho * (1.0 - cooled)
    dvdt = g * drho - (v ** 2) / L_cool * np.sign(v)
    # Decreasing buoyancy eventually halts tube and initiates overshoot.
    return [v, dvdt]


sol = solve_ivp(moving_tube_rhs, [0, 600], [-5e7, 0], dense_output=True,
                rtol=1e-6, max_step=1.0)
t_s = np.linspace(0, 600, 400)
h_km = sol.sol(t_s)[0] / 1e5
v_kms = sol.sol(t_s)[1] / 1e5

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(t_s, h_km, color='tab:blue')
ax[0].axhline(0, color='k', ls=':', lw=0.8)
ax[0].set_xlabel('time [s]'); ax[0].set_ylabel('tube height [km]')
ax[0].set_title('Tube apex rise and overshoot')
ax[0].grid(alpha=0.3)
ax[1].plot(t_s, v_kms, color='tab:red')
ax[1].axhline(0, color='k', ls=':', lw=0.8)
ax[1].set_xlabel('time [s]'); ax[1].set_ylabel('vertical velocity [km/s]')
ax[1].set_title('Wave-like oscillation → serpentine flow')
ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('Paper §3.4.2: tube undergoes rise, radiative cooling, serpentine oscillation.')

---
## 7. Magneto-convection regime map / 자기대류 체제 지도

Chandrasekhar number $\zeta = \eta/\kappa$. $\zeta<1$: oscillatory convection; $\zeta>1$: steady overturning. 논문 §3.5에 따르면, umbra 상위 2 Mm에서 $\zeta<1$이 실현되어 umbral dot이 oscillating magneto-convection으로 설명될 수 있다.

In [ ]:
def chandrasekhar_profile(z_Mm, eta_base=1e10, kappa_base=1e10):
    """Depth profile of zeta = eta/kappa.

    Simple model: eta decreases with depth (higher conductivity in deeper,
    denser plasma), kappa decreases faster → zeta drops below 1 near the
    surface and exceeds 1 deeper.

    Args:
        z_Mm: Depth in Mm (positive downward).
        eta_base: Reference magnetic diffusivity [cm^2/s].
        kappa_base: Reference thermal diffusivity [cm^2/s].
    """
    eta = eta_base * np.exp(-z_Mm / 4.0)
    kappa = kappa_base * np.exp(-z_Mm / 1.5)
    return eta / kappa


z_grid = np.linspace(0, 10, 200)
zeta = chandrasekhar_profile(z_grid)

plt.figure(figsize=(7.5, 5))
plt.semilogy(z_grid, zeta)
plt.axhline(1, color='k', ls='--', lw=0.8)
plt.fill_between(z_grid, 1e-2, 1, where=zeta < 1, alpha=0.2, color='tab:blue',
                 label=r'oscillatory ($\zeta<1$)')
plt.fill_between(z_grid, 1, 1e3, where=zeta >= 1, alpha=0.2, color='tab:red',
                 label=r'overturning ($\zeta>1$)')
plt.ylim(0.05, 100)
plt.xlabel('Depth [Mm]'); plt.ylabel(r'$\zeta = \eta/\kappa$')
plt.title('Magneto-convection regime (Weiss et al. 1990)')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

cross = z_grid[np.argmin(np.abs(zeta - 1))]
print(f'Transition oscillatory→overturning near z = {cross:.1f} Mm')
print('Paper: upper ~2 Mm of umbra has zeta<1 → supports oscillating umbral dots.')

---
## 8. Synthetic sunspot intensity map / 합성 흑점 강도 지도

Umbra (dark) + penumbra (filamentary) + quiet-Sun granulation을 결합한 toy model. 실제 MURaM 결과(Fig. 8, 13)의 거친 재현.

In [ ]:
def synthetic_sunspot(N=512, size_Mm=60.0, R_u=6.0, R_p=18.0,
                      n_filaments=160, rng=None):
    """Generate a toy intensity map of a sunspot + surroundings.

    Combines (i) an umbra core with ~19% intensity, (ii) a penumbral annulus
    with radially oriented filaments, and (iii) quiet-Sun granulation
    outside. Rough analogue of Figures 8/13 of the paper.

    Args:
        N: Image linear size in pixels.
        size_Mm: Physical size of the image [Mm].
        R_u: Umbra radius [Mm].
        R_p: Outer penumbra radius [Mm].
        n_filaments: Number of radial filaments to add.
        rng: Optional numpy Generator.

    Returns:
        2D float array with mean ≈ 1.0 (quiet Sun normalised).
    """
    if rng is None:
        rng = np.random.default_rng()
    y, x = np.mgrid[-N/2:N/2, -N/2:N/2]
    pix_Mm = size_Mm / N
    r = np.hypot(x, y) * pix_Mm
    phi = np.arctan2(y, x)

    # Granulation background.
    noise = rng.normal(size=(N, N))
    fx = np.fft.fftfreq(N)
    FX, FY = np.meshgrid(fx, fx)
    k = np.hypot(FX, FY)
    sigma = 1.0 / (2 * 5)
    lp = np.exp(-(k ** 2) / (2 * sigma ** 2))
    gran = np.real(np.fft.ifft2(np.fft.fft2(noise) * lp))
    gran = (gran - gran.mean()) / (gran.std() + 1e-12) * 0.13 + 1.0

    # Umbra: deep dark core.
    umbra_mask = r < R_u
    I = gran.copy()
    I[umbra_mask] = 0.19

    # Penumbra annulus + radial filaments.
    pen_mask = (r >= R_u) & (r < R_p)
    I[pen_mask] = 0.58 + 0.05 * rng.normal(size=pen_mask.sum())

    # Add radial filaments.
    for _ in range(n_filaments):
        phi0 = rng.uniform(-np.pi, np.pi)
        r0 = rng.uniform(R_u + 0.5, R_p - 0.5)
        width = rng.uniform(0.06, 0.20)
        bright = rng.uniform(0.75, 0.95)
        ang_dist = np.abs(((phi - phi0 + np.pi) % (2*np.pi)) - np.pi)
        rad_mask = (r >= R_u) & (r < R_p) & (ang_dist < 0.05)
        fil = np.exp(-0.5 * ((r - r0) / width) ** 2)
        I[rad_mask] = np.maximum(I[rad_mask], bright * fil[rad_mask] + 0.5 * (1 - fil[rad_mask]))
    # Dark cores in bright filaments.
    I[pen_mask] *= 1.0 + 0.03 * rng.normal(size=pen_mask.sum())
    return I


image = synthetic_sunspot(N=512, size_Mm=60.0, rng=rng)
plt.figure(figsize=(7, 7))
plt.imshow(image, cmap='gray', extent=[-30, 30, -30, 30], vmin=0.1, vmax=1.3)
plt.title('Synthetic sunspot intensity map (toy model)')
plt.xlabel('x [Mm]'); plt.ylabel('y [Mm]')
plt.colorbar(label='I / I_QS')
plt.show()

---
## 9. Evershed flow 프로파일 / Evershed flow radial profile

관측: inner penumbra ~1-2 km/s upflow, middle penumbra 지배적 horizontal ~5 km/s, outer penumbra downflow >5 km/s (국소 >10 km/s). 논문 Fig. 10, 11 기반.

In [ ]:
def evershed_profile(r_Mm, R_u=6.0, R_p=18.0):
    """Radial profiles of Evershed horizontal and vertical flow components.

    Very simplified empirical fit: horizontal flow peaks in mid penumbra,
    vertical flow transitions from upflow (inner) to downflow (outer).

    Args:
        r_Mm: Radial coordinate in Mm.
        R_u: Umbra radius [Mm].
        R_p: Outer penumbra radius [Mm].

    Returns:
        (v_horizontal, v_vertical) in km/s. Positive horizontal = outward.
    """
    x = (r_Mm - R_u) / (R_p - R_u)  # 0 at inner edge, 1 at outer
    x = np.clip(x, 0, 1)
    # Horizontal: rises to ~6.5 km/s at x~0.6, peaks at 8 km/s locally
    vH = 6.5 * np.sin(np.pi * x) + 2.0 * x
    vH[r_Mm < R_u] = 0  # no horizontal flow inside umbra in this model
    vH[r_Mm > R_p] = 0.5 * np.exp(-(r_Mm[r_Mm > R_p] - R_p) / 3) * 0  # no outside
    # Vertical: upflow (+) at x<0.3, downflow at x>0.7.
    vV = 1.5 * np.exp(-((x - 0.15) / 0.15) ** 2) - 3.0 * np.exp(-((x - 0.9) / 0.1) ** 2)
    vV[r_Mm < R_u] = 0
    vV[r_Mm > R_p] = 0
    return vH, vV


r_ev = np.linspace(0, 22, 400)
vH, vV = evershed_profile(r_ev)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(r_ev, vH, label='horizontal (Evershed)', color='tab:red', lw=2)
ax.plot(r_ev, vV, label='vertical (upflow/downflow)', color='tab:blue', lw=2)
ax.axvline(6, color='gray', ls='--', lw=0.8, label='umbra/penumbra')
ax.axvline(18, color='gray', ls=':', lw=0.8, label='penumbra/quiet')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('radius [Mm]')
ax.set_ylabel('velocity [km/s]')
ax.set_title('Evershed flow profile — empirical fit to §3.2.2 / Fig. 10-11')
ax.legend(); ax.grid(alpha=0.3)
plt.show()
print(f'Peak horizontal Evershed: {vH.max():.2f} km/s  (paper avg 6.5, local >10)')
print(f'Peak downflow in outer penumbra: {vV.min():.2f} km/s  (paper: often >5)')

---
## 10. Net Circular Polarization (NCP) 계산

NCP $= \int V(\lambda)\,d\lambda \ne 0$은 LOS 자기장 또는 속도의 gradient가 있을 때만 발생 (Sánchez Almeida & Lites 1992). 여기서는 2-component atmosphere (background + embedded flow channel)를 가정해 Stokes V를 forward-model하고 NCP를 계산한다.

In [ ]:
def stokes_V_two_component(wavelength, v_los1=0.0, v_los2=6.0, B1=1500.0,
                           B2=1500.0, filling=0.5, lambda_0=630.25,
                           doppler_width=0.03, g_eff=2.5):
    """Toy 2-component Stokes V for an 'uncombed' penumbra.

    Uses weak-field Gaussian line derivatives from two atmospheric
    components with different LOS velocities and field strengths.

    Args:
        wavelength: Wavelength grid [nm].
        v_los1, v_los2: LOS velocities of components 1 and 2 [km/s].
        B1, B2: LOS field strengths [G].
        filling: Filling factor of component 1 (0..1).
        lambda_0: Line rest wavelength [nm].
        doppler_width: Doppler width of Gaussian absorption [nm].
        g_eff: Effective Landé factor.

    Returns:
        Stokes V profile (relative intensity).
    """
    c = 3e5  # km/s
    lam1 = lambda_0 * (1 + v_los1 / c)
    lam2 = lambda_0 * (1 + v_los2 / c)
    # Zeeman splitting in nm (Larmor, weak-field approx)
    dl1 = 4.67e-13 * g_eff * B1 * (lambda_0 * 10) ** 2 / 10  # nm
    dl2 = 4.67e-13 * g_eff * B2 * (lambda_0 * 10) ** 2 / 10
    # Component profiles — derivative of Gaussian absorption.
    prof1 = np.exp(-((wavelength - lam1) / doppler_width) ** 2)
    prof2 = np.exp(-((wavelength - lam2) / doppler_width) ** 2)
    # V = -dlambda * d(I_abs)/d(lambda)
    dIdL1 = np.gradient(prof1, wavelength)
    dIdL2 = np.gradient(prof2, wavelength)
    V1 = -dl1 * dIdL1
    V2 = -dl2 * dIdL2
    return filling * V1 + (1 - filling) * V2


def _integrate(y, x):
    """Compatibility wrapper for numpy.trapezoid (numpy>=2) / numpy.trapz.\n"""
    if hasattr(np, 'trapezoid'):
        return np.trapezoid(y, x)
    return np.trapz(y, x)  # numpy<2 fallback


wl = np.linspace(630.15, 630.35, 400)
cases = {
    'Symmetric (single component)': dict(v_los1=0, v_los2=0, B1=1500, B2=1500, filling=0.5),
    'Uncombed (flow channel)':      dict(v_los1=0, v_los2=5, B1=1500, B2=1000, filling=0.5),
    'Uncombed + stronger field':    dict(v_los1=0, v_los2=5, B1=1500, B2=2000, filling=0.5),
    'Opposite polarity (downflow)': dict(v_los1=0, v_los2=-4, B1=1500, B2=-1500, filling=0.7),
}

fig, ax = plt.subplots(figsize=(8, 5))
for name, params in cases.items():
    V = stokes_V_two_component(wl, **params)
    ncp = _integrate(V, wl)
    ax.plot(wl, V, lw=1.5, label=f'{name}  NCP={ncp:.4f}')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel(r'$\lambda$ [nm]'); ax.set_ylabel('Stokes V (rel.)')
ax.set_title('Stokes V and NCP in uncombed penumbra (forward model)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('Nonzero NCP occurs only when LOS gradients of v or B exist — §3.2.2.')

---
## 11. Sunspot lifetime vs disconnection depth

Rempel (2011c): 16 Mm 깊이 domain에서 bottom BC 제약 없이 수명 1–2일. 50 Mm 깊이로 외삽 시 ~10일. 대류 timescale이 깊이에 따라 극적으로 증가하는 것이 핵심.
$$\tau_{\text{conv}}(z) \sim \frac{H_p(z)}{v_{\text{conv}}(z)},\qquad v_{\text{conv}}\propto\rho^{-1/3}\,F^{1/3}$$

In [ ]:
def convective_timescale(z_Mm, F_solar=6.3e10):
    """Approximate overturning convective timescale vs depth.

    Uses mixing-length scaling v ~ (F/rho)^(1/3) and tau = H_p / v.

    Args:
        z_Mm: Depth in Mm.
        F_solar: Solar energy flux [erg/cm^2/s].

    Returns:
        Convective timescale in days.
    """
    prof = convection_zone_profile(z_Mm * 1e8)
    v_conv = (F_solar / prof['rho']) ** (1/3)
    tau_s = prof['H_p'] / v_conv
    return tau_s / 86400.0


z_dep = np.logspace(-0.3, np.log10(100), 100)
tau_days = convective_timescale(z_dep)

plt.figure(figsize=(8, 5))
plt.loglog(z_dep, tau_days, lw=2)
plt.axvline(2, color='tab:red', ls='--', lw=0.8, label='helioseismic depth ~2 Mm')
plt.axvline(16, color='tab:orange', ls='--', lw=0.8, label='Rempel 2011c domain 16 Mm')
plt.axvline(50, color='tab:green', ls='--', lw=0.8, label='typical anchor 50 Mm')
plt.axhline(14, color='k', ls=':', lw=0.8, label='~2-week lifetime')
plt.xlabel('anchor depth [Mm]')
plt.ylabel('overturning timescale [days]')
plt.title('Sunspot lifetime scaling (Rempel 2011c)')
plt.legend(); plt.grid(which='both', alpha=0.3)
plt.show()

for zm in [2, 16, 50]:
    t = convective_timescale(np.array([zm]))[0]
    print(f'Anchor depth {zm:3d} Mm → overturning timescale {t:5.2f} d')

---
## 12. 결론 / Summary

이 노트북은 Rempel & Schlichenmaier (2011)의 핵심 물리를 11개 구현으로 재현했다:

1. **Convection-zone stratification** — polytropic model로 T, p, ρ 프로파일.
2. **Plasma β 깊이 의존성** — monolithic flux tube의 funneling 재현 (umbra 반경 2배 확장).
3. **Stefan-Boltzmann** — umbra 19%, penumbra 58% 광도 확인.
4. **Alfvén 속도와 CFL** — MHD sim의 timestep 제약 심각성 시각화.
5. **Jahn-Schmidt tripartite** — self-similar streamfunction으로 Fig. 1 재현.
6. **Moving flux tube ODE** — Schlichenmaier (1998) dynamic model 축약.
7. **Chandrasekhar number** — oscillatory/overturning regime 전환.
8. **Synthetic sunspot** — umbra + filamentary penumbra + granulation.
9. **Evershed flow** — 수평/수직 radial profile.
10. **NCP forward model** — 2-component uncombed field에서 nonzero NCP 생성.
11. **Lifetime scaling** — convective timescale의 깊이 의존성.

This notebook reproduces 11 key physical concepts of Rempel & Schlichenmaier (2011): stratification, depth-dependent plasma β, Stefan–Boltzmann luminosity, Alfvén-speed CFL constraint, the Jahn–Schmidt tripartite model, a simplified moving-flux-tube ODE, magneto-convection regimes, a synthetic sunspot intensity map, Evershed-flow radial profiles, a two-component NCP forward model, and a sunspot lifetime–depth scaling.

**다음 연구 방향 / Next directions**:
- Full MURaM-like MHD code (PENCIL, Dedalus, MURaM open source) 실행하여 penumbral filament 형성 재현
- DKIST SP/ViSP 데이터에 SIR/SNAPI Stokes inversion 적용 → observation-simulation comparison
- ML-based sunspot magnetic-field reconstruction from continuum images (본 노트북의 synthetic intensity와 NCP 모델이 학습 데이터 기반)
- Run a full open-source radiative MHD code to reproduce penumbral filament formation; apply Stokes inversions (SIR/SNAPI) to DKIST data for direct sim–obs comparison; train ML models to reconstruct sunspot magnetic fields from continuum images using the synthetic intensity/NCP generators developed here.